# Week 4: Linear Regression - Loss Function & Gradient Descent Visualization
# 第 4 週：線性回歸 — 損失函數、梯度下降視覺化

## 學習目標 Learning Objectives
1. 從零實作梯度下降 (Gradient Descent) 演算法
2. 繪製 3D 損失地形 (Loss Landscape) 視覺化
3. 實驗不同學習率 (Learning Rate) 對收斂的影響
4. 在等高線圖上描繪梯度下降軌跡動畫
5. 使用 sklearn 進行對比驗證
6. 進行殘差分析 (Residual Analysis)
7. 探索多項式回歸 (Polynomial Regression) 的過擬合 (Overfitting) 風險

---

In [ ]:
# 匯入必要套件 Import necessary packages
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import warnings
warnings.filterwarnings('ignore')

# 設定中文字型（如需要）
# plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'SimHei', 'Arial Unicode MS']
# plt.rcParams['axes.unicode_minus'] = False

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

print('All packages loaded successfully!')

---
## Part 1: Data Generation & Simple Linear Regression Intuition
## 第一部分：資料生成與簡單線性回歸的直覺

我們先產生合成資料 (Synthetic Data)，其真實關係為：

$$y = 3x + 7 + \epsilon, \quad \epsilon \sim \mathcal{N}(0, 5)$$

目標：用梯度下降找出 $w \approx 3$ 和 $b \approx 7$。

In [ ]:
# 設定隨機種子確保可重現性
np.random.seed(42)

# 真實參數 True parameters
w_true = 3.0
b_true = 7.0

# 產生資料 Generate data
n_samples = 100
X = np.random.uniform(0, 10, n_samples)
noise = np.random.normal(0, 5, n_samples)
y = w_true * X + b_true + noise

print(f'Number of samples: {n_samples}')
print(f'X range: [{X.min():.2f}, {X.max():.2f}]')
print(f'y range: [{y.min():.2f}, {y.max():.2f}]')
print(f'True parameters: w = {w_true}, b = {b_true}')

In [ ]:
# 散佈圖 Scatter plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(X, y, alpha=0.6, edgecolors='k', linewidth=0.5, label='Data points')

# 繪製真實直線 True line
x_line = np.linspace(0, 10, 100)
y_line = w_true * x_line + b_true
ax.plot(x_line, y_line, 'r--', linewidth=2, label=f'True: y = {w_true}x + {b_true}')

ax.set_xlabel('x (Feature)', fontsize=12)
ax.set_ylabel('y (Target)', fontsize=12)
ax.set_title('Simple Linear Regression - Synthetic Data', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Part 2: Implementing Gradient Descent from Scratch
## 第二部分：從零實作梯度下降

### MSE Loss Function
$$\mathcal{L}(w, b) = \frac{1}{n}\sum_{i=1}^{n}(y_i - wx_i - b)^2$$

### Gradients
$$\frac{\partial \mathcal{L}}{\partial w} = -\frac{2}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i) \cdot x_i$$

$$\frac{\partial \mathcal{L}}{\partial b} = -\frac{2}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)$$

### Update Rules
$$w \leftarrow w - \alpha \cdot \frac{\partial \mathcal{L}}{\partial w}$$
$$b \leftarrow b - \alpha \cdot \frac{\partial \mathcal{L}}{\partial b}$$

In [ ]:
def compute_mse(X, y, w, b):
    """Compute Mean Squared Error (MSE) loss.
    
    Parameters:
        X: Feature array (n,)
        y: Target array (n,)
        w: Weight (slope)
        b: Bias (intercept)
    Returns:
        MSE loss value (float)
    """
    y_pred = w * X + b
    mse = np.mean((y - y_pred) ** 2)
    return mse


def compute_gradients(X, y, w, b):
    """Compute gradients of MSE loss w.r.t. w and b.
    
    Parameters:
        X: Feature array (n,)
        y: Target array (n,)
        w: Weight (slope)
        b: Bias (intercept)
    Returns:
        dw: Gradient w.r.t. w
        db: Gradient w.r.t. b
    """
    n = len(X)
    y_pred = w * X + b
    error = y - y_pred  # Residuals
    
    dw = -(2 / n) * np.sum(error * X)
    db = -(2 / n) * np.sum(error)
    
    return dw, db


def gradient_descent(X, y, lr=0.01, epochs=100, w_init=0.0, b_init=0.0):
    """Run gradient descent for simple linear regression.
    
    Parameters:
        X: Feature array (n,)
        y: Target array (n,)
        lr: Learning rate (alpha)
        epochs: Number of iterations
        w_init: Initial weight
        b_init: Initial bias
    Returns:
        w_history: List of w values at each epoch
        b_history: List of b values at each epoch
        loss_history: List of MSE values at each epoch
    """
    w = w_init
    b = b_init
    
    w_history = [w]
    b_history = [b]
    loss_history = [compute_mse(X, y, w, b)]
    
    for epoch in range(epochs):
        # Step 1: Compute gradients
        dw, db = compute_gradients(X, y, w, b)
        
        # Step 2: Update parameters
        w = w - lr * dw
        b = b - lr * db
        
        # Step 3: Record history
        loss = compute_mse(X, y, w, b)
        w_history.append(w)
        b_history.append(b)
        loss_history.append(loss)
        
        # Clip to prevent overflow (for large learning rates)
        if np.isnan(loss) or loss > 1e10:
            print(f'  Divergence detected at epoch {epoch + 1}! Stopping.')
            break
    
    return w_history, b_history, loss_history


print('Functions defined successfully!')

In [ ]:
# 執行梯度下降 Run gradient descent
lr = 0.01
epochs = 200

w_hist, b_hist, loss_hist = gradient_descent(X, y, lr=lr, epochs=epochs)

w_final = w_hist[-1]
b_final = b_hist[-1]
loss_final = loss_hist[-1]

print(f'Learning rate: {lr}')
print(f'Epochs: {epochs}')
print(f'---')
print(f'Final w = {w_final:.4f}  (True: {w_true})')
print(f'Final b = {b_final:.4f}  (True: {b_true})')
print(f'Final MSE = {loss_final:.4f}')

In [ ]:
# 視覺化結果 Visualize results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Loss curve
axes[0].plot(loss_hist, 'b-', linewidth=1.5)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('MSE Loss', fontsize=12)
axes[0].set_title('Loss Curve (MSE vs Epochs)', fontsize=13)
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(0, epochs)

# Right: Scatter + fitted line
axes[1].scatter(X, y, alpha=0.6, edgecolors='k', linewidth=0.5, label='Data')
x_line = np.linspace(0, 10, 100)
axes[1].plot(x_line, w_true * x_line + b_true, 'r--', linewidth=2,
             label=f'True: y = {w_true}x + {b_true}')
axes[1].plot(x_line, w_final * x_line + b_final, 'g-', linewidth=2,
             label=f'GD: y = {w_final:.2f}x + {b_final:.2f}')
axes[1].set_xlabel('x', fontsize=12)
axes[1].set_ylabel('y', fontsize=12)
axes[1].set_title('Fitted Line vs True Line', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Part 3: 3D Loss Landscape Visualization
## 第三部分：損失地形 3D 視覺化

我們將 MSE 損失函數畫成 $(w, b)$ 空間中的 3D 曲面。
由於 MSE 對線性回歸是凸函數 (Convex Function)，損失地形呈碗狀 (Bowl-shaped)，
碗底即為全域最小值 (Global Minimum)。

In [ ]:
def compute_loss_surface(X, y, w_range, b_range, resolution=100):
    """Compute the MSE loss surface over a grid of (w, b) values.
    
    Parameters:
        X, y: Training data
        w_range: (w_min, w_max) tuple
        b_range: (b_min, b_max) tuple
        resolution: Number of grid points per axis
    Returns:
        W, B: Meshgrid arrays
        L: Loss values at each grid point
    """
    w_vals = np.linspace(w_range[0], w_range[1], resolution)
    b_vals = np.linspace(b_range[0], b_range[1], resolution)
    W, B = np.meshgrid(w_vals, b_vals)
    L = np.zeros_like(W)
    
    for i in range(resolution):
        for j in range(resolution):
            L[i, j] = compute_mse(X, y, W[i, j], B[i, j])
    
    return W, B, L


# 計算損失曲面 Compute loss surface
w_range = (-2, 8)
b_range = (-5, 20)
W_grid, B_grid, L_grid = compute_loss_surface(X, y, w_range, b_range, resolution=100)

print(f'Loss surface computed: {L_grid.shape}')
print(f'Min loss on grid: {L_grid.min():.4f} at w={W_grid.ravel()[L_grid.argmin()]:.2f}, b={B_grid.ravel()[L_grid.argmin()]:.2f}')

In [ ]:
# 3D 損失地形圖 3D Loss Landscape
fig = plt.figure(figsize=(14, 6))

# Left: 3D surface plot
ax1 = fig.add_subplot(121, projection='3d')
surf = ax1.plot_surface(W_grid, B_grid, L_grid, cmap=cm.viridis, alpha=0.8,
                        edgecolor='none', antialiased=True)
ax1.set_xlabel('w (Weight)', fontsize=10)
ax1.set_ylabel('b (Bias)', fontsize=10)
ax1.set_zlabel('MSE Loss', fontsize=10)
ax1.set_title('3D Loss Landscape', fontsize=13)
ax1.view_init(elev=30, azim=225)
fig.colorbar(surf, ax=ax1, shrink=0.5, aspect=10, pad=0.1)

# Plot the optimal point
ax1.scatter([w_final], [b_final], [loss_final], color='red', s=100, zorder=5,
            label='GD Solution')

# Right: Contour plot
ax2 = fig.add_subplot(122)
contour = ax2.contour(W_grid, B_grid, L_grid, levels=30, cmap=cm.viridis)
ax2.clabel(contour, inline=True, fontsize=7)
ax2.set_xlabel('w (Weight)', fontsize=12)
ax2.set_ylabel('b (Bias)', fontsize=12)
ax2.set_title('Contour Plot (Top View of Loss Landscape)', fontsize=13)

# Plot GD trajectory
ax2.plot(w_hist, b_hist, 'r.-', markersize=2, linewidth=1, alpha=0.7, label='GD Trajectory')
ax2.plot(w_hist[0], b_hist[0], 'go', markersize=10, label='Start')
ax2.plot(w_hist[-1], b_hist[-1], 'r*', markersize=15, label='End')
ax2.plot(w_true, b_true, 'bx', markersize=12, markeredgewidth=3, label='True (w=3, b=7)')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**觀察重點 Key Observations:**
1. 損失地形確實呈碗狀 — MSE 是凸函數，保證唯一全域最小值
2. 等高線越密集的區域，梯度越大（地形越陡峭）
3. 梯度下降的軌跡從起始點出發，沿著損失下降最快的方向逐步收斂到碗底

---

## Part 4: Learning Rate Experiment
## 第四部分：學習率實驗

測試四種學習率 $\alpha \in \{0.001, 0.01, 0.1, 1.0\}$，觀察收斂行為的差異。

- **太小 (0.001):** 收斂極慢
- **適中 (0.01):** 穩定收斂
- **較大 (0.1):** 快速但可能震盪
- **太大 (1.0):** 發散！

In [ ]:
# 學習率實驗 Learning Rate Experiment
learning_rates = [0.001, 0.01, 0.1, 1.0]
colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']
labels = ['lr=0.001 (Too small)', 'lr=0.01 (Good)', 'lr=0.1 (Fast)', 'lr=1.0 (Too large)']
epochs_exp = 200

results = {}
for lr_val in learning_rates:
    w_h, b_h, l_h = gradient_descent(X, y, lr=lr_val, epochs=epochs_exp)
    results[lr_val] = {'w': w_h, 'b': b_h, 'loss': l_h}
    status = 'Converged' if l_h[-1] < 100 else 'Diverged'
    print(f'lr={lr_val:<6} | Final MSE={l_h[-1]:>12.4f} | w={w_h[-1]:>8.4f} | b={b_h[-1]:>8.4f} | {status}')

In [ ]:
# 損失曲線比較 Loss Curve Comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: All learning rates (full scale)
for i, lr_val in enumerate(learning_rates):
    loss_data = results[lr_val]['loss']
    # Clip for visualization when diverged
    loss_clipped = [min(l, 500) for l in loss_data]
    axes[0].plot(loss_clipped, color=colors[i], linewidth=2, label=labels[i])

axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('MSE Loss', fontsize=12)
axes[0].set_title('Learning Rate Comparison (Full Scale)', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 500)

# Right: Converged learning rates only (zoomed in)
for i, lr_val in enumerate(learning_rates[:3]):  # Exclude lr=1.0
    axes[1].plot(results[lr_val]['loss'], color=colors[i], linewidth=2, label=labels[i])

axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('MSE Loss', fontsize=12)
axes[1].set_title('Learning Rate Comparison (Zoomed, excluding lr=1.0)', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 等高線圖上的學習率軌跡對比 Contour plot with trajectories
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for idx, (lr_val, ax) in enumerate(zip(learning_rates, axes.ravel())):
    res = results[lr_val]
    
    # Draw contour
    contour = ax.contour(W_grid, B_grid, L_grid, levels=25, cmap=cm.viridis, alpha=0.7)
    ax.contourf(W_grid, B_grid, L_grid, levels=25, cmap=cm.viridis, alpha=0.3)
    
    # Draw trajectory (limit to visible range)
    w_traj = np.array(res['w'])
    b_traj = np.array(res['b'])
    
    # Clip to visible range
    mask = (w_traj >= w_range[0]) & (w_traj <= w_range[1]) & \
           (b_traj >= b_range[0]) & (b_traj <= b_range[1])
    
    if mask.sum() > 1:
        w_vis = w_traj[mask]
        b_vis = b_traj[mask]
        ax.plot(w_vis, b_vis, 'r.-', markersize=3, linewidth=1.2, alpha=0.8)
        ax.plot(w_vis[0], b_vis[0], 'go', markersize=10, zorder=5, label='Start')
        ax.plot(w_vis[-1], b_vis[-1], 'r*', markersize=15, zorder=5, label='End')
    else:
        ax.plot(res['w'][0], res['b'][0], 'go', markersize=10, label='Start')
        ax.text(0.5, 0.5, 'DIVERGED!', transform=ax.transAxes,
                fontsize=20, color='red', ha='center', va='center', fontweight='bold')
    
    ax.plot(w_true, b_true, 'bx', markersize=12, markeredgewidth=3, label=f'True ({w_true},{b_true})')
    ax.set_xlabel('w', fontsize=11)
    ax.set_ylabel('b', fontsize=11)
    ax.set_title(f'lr = {lr_val}', fontsize=13, fontweight='bold', color=colors[idx])
    ax.legend(fontsize=8, loc='upper right')
    ax.set_xlim(w_range)
    ax.set_ylim(b_range)
    ax.grid(True, alpha=0.2)

plt.suptitle('Gradient Descent Trajectories for Different Learning Rates',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### Analysis of Learning Rate Effects
### 學習率影響分析

| Learning Rate | Behavior | Observation |
|:---:|:---:|:---:|
| 0.001 | Converges very slowly | Trajectory barely moves from start, needs many more epochs |
| 0.01 | Stable convergence | Smooth path toward the optimum |
| 0.1 | Fast but may oscillate | Reaches near-optimum quickly, slight zigzag possible |
| 1.0 | Diverges | Parameters explode, loss increases without bound |

---

## Part 5: Gradient Descent Trajectory Animation
## 第五部分：梯度下降軌跡動畫

在等高線圖上動態呈現梯度下降的每一步移動。

In [ ]:
# 梯度下降動畫 Gradient Descent Animation
# Using lr=0.01 for clear visualization
w_anim, b_anim, loss_anim = gradient_descent(X, y, lr=0.01, epochs=100)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Left: Contour plot with animated trajectory
ax1.contourf(W_grid, B_grid, L_grid, levels=30, cmap=cm.viridis, alpha=0.4)
contour = ax1.contour(W_grid, B_grid, L_grid, levels=30, cmap=cm.viridis, alpha=0.6)
ax1.plot(w_true, b_true, 'bx', markersize=12, markeredgewidth=3, label='True')

trajectory_line, = ax1.plot([], [], 'r-', linewidth=1.5, alpha=0.8)
current_point, = ax1.plot([], [], 'ro', markersize=10, zorder=5)
start_point, = ax1.plot([w_anim[0]], [b_anim[0]], 'go', markersize=10, label='Start')

ax1.set_xlabel('w (Weight)', fontsize=12)
ax1.set_ylabel('b (Bias)', fontsize=12)
ax1.set_title('Gradient Descent Trajectory (lr=0.01)', fontsize=13)
ax1.set_xlim(w_range)
ax1.set_ylim(b_range)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.2)

# Right: Loss curve
loss_line, = ax2.plot([], [], 'b-', linewidth=2)
loss_point, = ax2.plot([], [], 'ro', markersize=8, zorder=5)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('MSE Loss', fontsize=12)
ax2.set_title('Loss Curve', fontsize=13)
ax2.set_xlim(0, len(loss_anim))
ax2.set_ylim(0, max(loss_anim) * 1.1)
ax2.grid(True, alpha=0.3)

epoch_text = ax1.text(0.02, 0.98, '', transform=ax1.transAxes,
                      fontsize=11, verticalalignment='top',
                      bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

def init():
    trajectory_line.set_data([], [])
    current_point.set_data([], [])
    loss_line.set_data([], [])
    loss_point.set_data([], [])
    epoch_text.set_text('')
    return trajectory_line, current_point, loss_line, loss_point, epoch_text

def animate(frame):
    # Update trajectory
    trajectory_line.set_data(w_anim[:frame+1], b_anim[:frame+1])
    current_point.set_data([w_anim[frame]], [b_anim[frame]])
    
    # Update loss curve
    loss_line.set_data(range(frame+1), loss_anim[:frame+1])
    loss_point.set_data([frame], [loss_anim[frame]])
    
    # Update text
    epoch_text.set_text(f'Epoch: {frame}\nw={w_anim[frame]:.3f}\nb={b_anim[frame]:.3f}\nLoss={loss_anim[frame]:.2f}')
    
    return trajectory_line, current_point, loss_line, loss_point, epoch_text

anim = FuncAnimation(fig, animate, init_func=init, frames=len(w_anim),
                     interval=100, blit=True, repeat=False)

plt.tight_layout()

# Display animation in notebook
HTML(anim.to_jshtml())

**Alternative: Static version showing trajectory steps with arrows**

In [ ]:
# 靜態版本：帶箭頭的梯度下降軌跡 Static version with arrows
fig, ax = plt.subplots(figsize=(10, 8))

ax.contourf(W_grid, B_grid, L_grid, levels=30, cmap=cm.viridis, alpha=0.4)
contour = ax.contour(W_grid, B_grid, L_grid, levels=30, cmap=cm.viridis, alpha=0.6)
ax.clabel(contour, inline=True, fontsize=7)

# Draw trajectory with arrows (show every 5th step for clarity)
step = 5
for i in range(0, min(len(w_anim)-1, 100), step):
    dw = w_anim[i+1] - w_anim[i]
    db = b_anim[i+1] - b_anim[i]
    ax.annotate('', xy=(w_anim[i+1], b_anim[i+1]),
                xytext=(w_anim[i], b_anim[i]),
                arrowprops=dict(arrowstyle='->', color='red', lw=1.5, alpha=0.7))

# Mark special points
ax.plot(w_anim[0], b_anim[0], 'go', markersize=12, zorder=5, label='Start (0, 0)')
ax.plot(w_anim[-1], b_anim[-1], 'r*', markersize=18, zorder=5,
        label=f'End ({w_anim[-1]:.2f}, {b_anim[-1]:.2f})')
ax.plot(w_true, b_true, 'bx', markersize=14, markeredgewidth=3,
        label=f'True ({w_true}, {b_true})')

ax.set_xlabel('w (Weight)', fontsize=13)
ax.set_ylabel('b (Bias)', fontsize=13)
ax.set_title('Gradient Descent Trajectory on Contour Plot (lr=0.01)', fontsize=14)
ax.legend(fontsize=11, loc='upper right')
ax.set_xlim(w_range)
ax.set_ylim(b_range)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

---
## Part 6: Comparison with sklearn LinearRegression
## 第六部分：與 sklearn 的對比驗證

我們用 sklearn 的 `LinearRegression`（使用正規方程 Normal Equation）來驗證我們的手動實作結果。

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# sklearn LinearRegression (uses Normal Equation)
model_sklearn = LinearRegression()
model_sklearn.fit(X.reshape(-1, 1), y)

w_sklearn = model_sklearn.coef_[0]
b_sklearn = model_sklearn.intercept_
y_pred_sklearn = model_sklearn.predict(X.reshape(-1, 1))
mse_sklearn = mean_squared_error(y, y_pred_sklearn)

print('=== Comparison: Manual GD vs sklearn ===')
print(f'{"":<15} {"Manual GD":>12} {"sklearn":>12} {"True":>12}')
print(f'{"w (weight)":<15} {w_final:>12.4f} {w_sklearn:>12.4f} {w_true:>12.4f}')
print(f'{"b (bias)":<15} {b_final:>12.4f} {b_sklearn:>12.4f} {b_true:>12.4f}')
print(f'{"MSE":<15} {loss_final:>12.4f} {mse_sklearn:>12.4f} {"---":>12}')
print()
print(f'Difference in w: {abs(w_final - w_sklearn):.6f}')
print(f'Difference in b: {abs(b_final - b_sklearn):.6f}')

In [ ]:
# 視覺化比較 Visual comparison
fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(X, y, alpha=0.5, edgecolors='k', linewidth=0.5, label='Data', zorder=2)

x_line = np.linspace(0, 10, 100)
ax.plot(x_line, w_true * x_line + b_true, 'r--', linewidth=2, label='True', zorder=3)
ax.plot(x_line, w_final * x_line + b_final, 'g-', linewidth=2.5,
        label=f'Manual GD (w={w_final:.3f}, b={b_final:.3f})', zorder=4)
ax.plot(x_line, w_sklearn * x_line + b_sklearn, 'b:', linewidth=2.5,
        label=f'sklearn (w={w_sklearn:.3f}, b={b_sklearn:.3f})', zorder=4)

ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('y', fontsize=12)
ax.set_title('Manual Gradient Descent vs sklearn LinearRegression', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Part 7: Residual Analysis
## 第七部分：殘差分析

殘差 (Residual) = 真實值 - 預測值：$e_i = y_i - \hat{y}_i$

健康的殘差應該：
1. 隨機分散在零線附近（沒有模式）
2. 接近常態分布 (Normal Distribution)
3. 變異性恆定（同質變異性, Homoscedasticity）

In [ ]:
from scipy import stats

# 計算殘差 Compute residuals
y_pred = w_final * X + b_final
residuals = y - y_pred

# 計算評估指標 Compute evaluation metrics
mse_val = mean_squared_error(y, y_pred)
rmse_val = np.sqrt(mse_val)
mae_val = mean_absolute_error(y, y_pred)
r2_val = r2_score(y, y_pred)
n_features = 1
adj_r2_val = 1 - (1 - r2_val) * (n_samples - 1) / (n_samples - n_features - 1)

print('=== Regression Evaluation Metrics ===')
print(f'MSE  (Mean Squared Error):     {mse_val:.4f}')
print(f'RMSE (Root Mean Squared Error): {rmse_val:.4f}')
print(f'MAE  (Mean Absolute Error):     {mae_val:.4f}')
print(f'R^2  (R-squared):               {r2_val:.4f}')
print(f'Adj R^2 (Adjusted R-squared):   {adj_r2_val:.4f}')
print()
print(f'Interpretation: The model explains {r2_val*100:.1f}% of the variance in y.')

In [ ]:
# 殘差分析圖 Residual Analysis Plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Residuals vs Fitted Values
axes[0, 0].scatter(y_pred, residuals, alpha=0.6, edgecolors='k', linewidth=0.5)
axes[0, 0].axhline(y=0, color='r', linestyle='--', linewidth=1.5)
axes[0, 0].set_xlabel('Fitted Values (Predicted y)', fontsize=11)
axes[0, 0].set_ylabel('Residuals', fontsize=11)
axes[0, 0].set_title('Residuals vs Fitted Values', fontsize=13)
axes[0, 0].grid(True, alpha=0.3)

# 2. Histogram of Residuals
axes[0, 1].hist(residuals, bins=20, edgecolor='black', alpha=0.7, density=True,
                color='steelblue', label='Residuals')
# Overlay normal distribution
x_norm = np.linspace(residuals.min(), residuals.max(), 100)
axes[0, 1].plot(x_norm, stats.norm.pdf(x_norm, residuals.mean(), residuals.std()),
                'r-', linewidth=2, label='Normal Distribution')
axes[0, 1].set_xlabel('Residual Value', fontsize=11)
axes[0, 1].set_ylabel('Density', fontsize=11)
axes[0, 1].set_title('Distribution of Residuals', fontsize=13)
axes[0, 1].legend(fontsize=10)
axes[0, 1].grid(True, alpha=0.3)

# 3. Q-Q Plot
stats.probplot(residuals, dist='norm', plot=axes[1, 0])
axes[1, 0].set_title('Q-Q Plot (Normal Probability Plot)', fontsize=13)
axes[1, 0].grid(True, alpha=0.3)

# 4. Residuals vs X (to check for patterns)
axes[1, 1].scatter(X, residuals, alpha=0.6, edgecolors='k', linewidth=0.5)
axes[1, 1].axhline(y=0, color='r', linestyle='--', linewidth=1.5)
axes[1, 1].set_xlabel('X (Feature)', fontsize=11)
axes[1, 1].set_ylabel('Residuals', fontsize=11)
axes[1, 1].set_title('Residuals vs Feature X', fontsize=13)
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('Residual Analysis Dashboard', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# Shapiro-Wilk test for normality
stat, p_value = stats.shapiro(residuals)
print(f'\nShapiro-Wilk Normality Test:')
print(f'  Statistic = {stat:.4f}')
print(f'  p-value   = {p_value:.4f}')
print(f'  Conclusion: {"Residuals appear normal (p > 0.05)" if p_value > 0.05 else "Residuals may not be normal (p <= 0.05)"}')

---
## Part 8: Polynomial Regression & Overfitting Demo
## 第八部分：多項式回歸與過擬合示範

比較 degree = 1 (linear), 3, 10 的擬合效果。

- **degree=1:** 欠擬合 (Underfitting) — 太簡單，遺漏非線性趨勢
- **degree=3:** 適當擬合 (Good Fit) — 平滑追蹤資料趨勢
- **degree=10:** 過擬合 (Overfitting) — 穿過每個點，劇烈扭曲

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split

# 產生具有非線性關係的資料 Generate nonlinear data
np.random.seed(42)
n_poly = 30
X_poly = np.sort(np.random.uniform(0, 1, n_poly))
y_poly = np.sin(2 * np.pi * X_poly) + np.random.normal(0, 0.3, n_poly)

# Split into train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_poly, y_poly, test_size=0.3, random_state=42
)

# Sort for plotting
sort_idx_train = np.argsort(X_train)
sort_idx_test = np.argsort(X_test)

print(f'Training samples: {len(X_train)}')
print(f'Test samples: {len(X_test)}')

In [ ]:
# 多項式回歸比較 Polynomial Regression Comparison
degrees = [1, 3, 10]
colors_poly = ['#2196F3', '#4CAF50', '#F44336']
titles = ['Degree 1 (Underfitting)', 'Degree 3 (Good Fit)', 'Degree 10 (Overfitting)']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

x_smooth = np.linspace(0, 1, 300)
y_true_curve = np.sin(2 * np.pi * x_smooth)

train_mses = []
test_mses = []

for idx, (degree, ax) in enumerate(zip(degrees, axes)):
    # Fit polynomial regression
    model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    model.fit(X_train.reshape(-1, 1), y_train)
    
    # Predict
    y_train_pred = model.predict(X_train.reshape(-1, 1))
    y_test_pred = model.predict(X_test.reshape(-1, 1))
    y_smooth = model.predict(x_smooth.reshape(-1, 1))
    
    # Compute MSE
    train_mse = mean_squared_error(y_train, y_train_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)
    train_mses.append(train_mse)
    test_mses.append(test_mse)
    
    # Plot
    ax.scatter(X_train, y_train, c='blue', s=40, alpha=0.7, edgecolors='k',
               linewidth=0.5, label='Train', zorder=3)
    ax.scatter(X_test, y_test, c='orange', s=40, alpha=0.7, edgecolors='k',
               linewidth=0.5, marker='^', label='Test', zorder=3)
    ax.plot(x_smooth, y_true_curve, 'k--', linewidth=1, alpha=0.5, label='True function')
    ax.plot(x_smooth, y_smooth, color=colors_poly[idx], linewidth=2.5,
            label=f'Poly deg={degree}')
    
    ax.set_xlabel('x', fontsize=11)
    ax.set_ylabel('y', fontsize=11)
    ax.set_title(f'{titles[idx]}\nTrain MSE={train_mse:.4f}, Test MSE={test_mse:.4f}',
                 fontsize=12)
    ax.legend(fontsize=8, loc='upper right')
    ax.set_ylim(-2, 2)
    ax.grid(True, alpha=0.3)

plt.suptitle('Polynomial Regression: Underfitting vs Good Fit vs Overfitting',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 不同 degree 的 Train/Test MSE 比較 Degree vs MSE plot
degrees_all = list(range(1, 16))
train_mses_all = []
test_mses_all = []

for deg in degrees_all:
    model = make_pipeline(PolynomialFeatures(deg), LinearRegression())
    model.fit(X_train.reshape(-1, 1), y_train)
    train_mses_all.append(mean_squared_error(y_train, model.predict(X_train.reshape(-1, 1))))
    test_mse = mean_squared_error(y_test, model.predict(X_test.reshape(-1, 1)))
    test_mses_all.append(min(test_mse, 5))  # Clip for visualization

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(degrees_all, train_mses_all, 'bo-', linewidth=2, markersize=8, label='Train MSE')
ax.plot(degrees_all, test_mses_all, 'r^-', linewidth=2, markersize=8, label='Test MSE')

# Highlight the sweet spot
best_deg = degrees_all[np.argmin(test_mses_all)]
ax.axvline(x=best_deg, color='green', linestyle='--', alpha=0.5,
           label=f'Best degree = {best_deg}')

# Annotate regions
ax.annotate('Underfitting\nZone', xy=(1.5, max(test_mses_all)*0.8),
            fontsize=11, color='blue', ha='center')
ax.annotate('Overfitting\nZone', xy=(12, max(test_mses_all)*0.8),
            fontsize=11, color='red', ha='center')

ax.set_xlabel('Polynomial Degree', fontsize=13)
ax.set_ylabel('MSE', fontsize=13)
ax.set_title('Bias-Variance Tradeoff: Polynomial Degree vs MSE', fontsize=14)
ax.legend(fontsize=11)
ax.set_xticks(degrees_all)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Key Takeaways from Polynomial Regression
### 多項式回歸的關鍵觀察

1. **Train MSE 隨 degree 增加而持續下降** — 更複雜的模型總是能更好地擬合訓練資料
2. **Test MSE 先降後升** — 這就是偏差-變異權衡 (Bias-Variance Tradeoff)
3. **最佳 degree** 是在 Test MSE 最低的那個點
4. **應對過擬合：** 交叉驗證、正則化 (Ridge/Lasso)、增加訓練資料

---

## Part 9: Loss Function Comparison (MSE vs MAE vs Huber)
## 第九部分：損失函數比較

In [ ]:
# 損失函數形狀比較 Loss function shape comparison
residual_range = np.linspace(-5, 5, 500)

mse_loss = residual_range ** 2
mae_loss = np.abs(residual_range)

delta = 1.0
huber_loss = np.where(
    np.abs(residual_range) <= delta,
    0.5 * residual_range ** 2,
    delta * np.abs(residual_range) - 0.5 * delta ** 2
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss function shapes
ax1.plot(residual_range, mse_loss, 'b-', linewidth=2.5, label='MSE (Squared Error)')
ax1.plot(residual_range, mae_loss, 'r-', linewidth=2.5, label='MAE (Absolute Error)')
ax1.plot(residual_range, huber_loss, 'g-', linewidth=2.5, label=f'Huber (delta={delta})')
ax1.set_xlabel('Residual (y - y_hat)', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Loss Function Comparison', fontsize=13)
ax1.legend(fontsize=11)
ax1.set_ylim(0, 10)
ax1.grid(True, alpha=0.3)

# Gradient shapes
mse_grad = 2 * residual_range
mae_grad = np.sign(residual_range)
huber_grad = np.where(
    np.abs(residual_range) <= delta,
    residual_range,
    delta * np.sign(residual_range)
)

ax2.plot(residual_range, mse_grad, 'b-', linewidth=2.5, label='MSE Gradient')
ax2.plot(residual_range, mae_grad, 'r-', linewidth=2.5, label='MAE Gradient')
ax2.plot(residual_range, huber_grad, 'g-', linewidth=2.5, label='Huber Gradient')
ax2.set_xlabel('Residual (y - y_hat)', fontsize=12)
ax2.set_ylabel('Gradient', fontsize=12)
ax2.set_title('Loss Function Gradients', fontsize=13)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)
ax2.axhline(y=0, color='k', linewidth=0.5)

plt.tight_layout()
plt.show()

---
## Part 10: Batch GD vs SGD vs Mini-batch GD
## 第十部分：三種梯度下降變體的比較

In [ ]:
def sgd(X, y, lr=0.01, epochs=100, w_init=0.0, b_init=0.0):
    """Stochastic Gradient Descent: Update with ONE random sample per step."""
    w, b = w_init, b_init
    n = len(X)
    w_history, b_history, loss_history = [w], [b], [compute_mse(X, y, w, b)]
    
    for epoch in range(epochs):
        # Shuffle data
        indices = np.random.permutation(n)
        for i in indices:
            xi, yi = X[i], y[i]
            y_pred = w * xi + b
            error = yi - y_pred
            dw = -2 * error * xi
            db = -2 * error
            w = w - lr * dw
            b = b - lr * db
        
        loss = compute_mse(X, y, w, b)
        w_history.append(w)
        b_history.append(b)
        loss_history.append(loss)
    
    return w_history, b_history, loss_history


def minibatch_gd(X, y, lr=0.01, epochs=100, batch_size=16, w_init=0.0, b_init=0.0):
    """Mini-batch Gradient Descent."""
    w, b = w_init, b_init
    n = len(X)
    w_history, b_history, loss_history = [w], [b], [compute_mse(X, y, w, b)]
    
    for epoch in range(epochs):
        indices = np.random.permutation(n)
        for start in range(0, n, batch_size):
            batch_idx = indices[start:start + batch_size]
            X_batch, y_batch = X[batch_idx], y[batch_idx]
            m = len(X_batch)
            y_pred = w * X_batch + b
            error = y_batch - y_pred
            dw = -(2 / m) * np.sum(error * X_batch)
            db = -(2 / m) * np.sum(error)
            w = w - lr * dw
            b = b - lr * db
        
        loss = compute_mse(X, y, w, b)
        w_history.append(w)
        b_history.append(b)
        loss_history.append(loss)
    
    return w_history, b_history, loss_history


# Run all three methods
np.random.seed(42)
lr_compare = 0.005
epochs_compare = 50

w_bgd, b_bgd, l_bgd = gradient_descent(X, y, lr=lr_compare, epochs=epochs_compare)
w_sgd, b_sgd, l_sgd = sgd(X, y, lr=0.001, epochs=epochs_compare)  # SGD needs smaller lr
w_mbgd, b_mbgd, l_mbgd = minibatch_gd(X, y, lr=lr_compare, epochs=epochs_compare, batch_size=16)

print(f'BGD  final: w={w_bgd[-1]:.4f}, b={b_bgd[-1]:.4f}, MSE={l_bgd[-1]:.4f}')
print(f'SGD  final: w={w_sgd[-1]:.4f}, b={b_sgd[-1]:.4f}, MSE={l_sgd[-1]:.4f}')
print(f'MBGD final: w={w_mbgd[-1]:.4f}, b={b_mbgd[-1]:.4f}, MSE={l_mbgd[-1]:.4f}')

In [ ]:
# 視覺化比較 Visual comparison of three GD variants
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Left: Loss curves
ax1.plot(l_bgd, 'b-', linewidth=2, label='Batch GD', alpha=0.9)
ax1.plot(l_sgd, 'r-', linewidth=1.5, label='SGD', alpha=0.7)
ax1.plot(l_mbgd, 'g-', linewidth=2, label='Mini-batch GD (bs=16)', alpha=0.9)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('MSE Loss', fontsize=12)
ax1.set_title('Loss Curves: BGD vs SGD vs Mini-batch', fontsize=13)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Right: Trajectories on contour plot
ax2.contourf(W_grid, B_grid, L_grid, levels=25, cmap=cm.viridis, alpha=0.3)
ax2.contour(W_grid, B_grid, L_grid, levels=25, cmap=cm.viridis, alpha=0.5)

ax2.plot(w_bgd, b_bgd, 'b.-', markersize=3, linewidth=1.5, alpha=0.8, label='Batch GD')
ax2.plot(w_sgd, b_sgd, 'r.-', markersize=3, linewidth=1, alpha=0.6, label='SGD')
ax2.plot(w_mbgd, b_mbgd, 'g.-', markersize=3, linewidth=1.5, alpha=0.8, label='Mini-batch')

ax2.plot(0, 0, 'ko', markersize=10, label='Start', zorder=5)
ax2.plot(w_true, b_true, 'mx', markersize=14, markeredgewidth=3, label='True', zorder=5)

ax2.set_xlabel('w', fontsize=12)
ax2.set_ylabel('b', fontsize=12)
ax2.set_title('Trajectories: BGD vs SGD vs Mini-batch', fontsize=13)
ax2.legend(fontsize=9)
ax2.set_xlim(w_range)
ax2.set_ylim(b_range)
ax2.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

### Observations
### 觀察結果

- **Batch GD (藍):** 軌跡最平滑，每步方向一致，但速度取決於學習率
- **SGD (紅):** 軌跡最不穩定，由於每步只用一個樣本，方向噪聲大
- **Mini-batch (綠):** 兼顧兩者，軌跡有些微震盪但整體方向正確

---

## Exercises
## 練習題

### Exercise 1: Gradient Verification
用數值微分 (Numerical Differentiation) 驗證你的梯度計算。

提示：數值梯度 $\approx \frac{\mathcal{L}(w + \epsilon) - \mathcal{L}(w - \epsilon)}{2\epsilon}$，其中 $\epsilon = 10^{-5}$

In [ ]:
# Exercise 1: Verify gradients numerically
epsilon = 1e-5
w_test, b_test = 1.0, 2.0

# Analytical gradients (your implementation)
dw_analytical, db_analytical = compute_gradients(X, y, w_test, b_test)

# Numerical gradients
dw_numerical = (compute_mse(X, y, w_test + epsilon, b_test) -
                compute_mse(X, y, w_test - epsilon, b_test)) / (2 * epsilon)
db_numerical = (compute_mse(X, y, w_test, b_test + epsilon) -
                compute_mse(X, y, w_test, b_test - epsilon)) / (2 * epsilon)

print('Gradient Verification:')
print(f'  dw: Analytical = {dw_analytical:.8f}, Numerical = {dw_numerical:.8f}, '
      f'Diff = {abs(dw_analytical - dw_numerical):.2e}')
print(f'  db: Analytical = {db_analytical:.8f}, Numerical = {db_numerical:.8f}, '
      f'Diff = {abs(db_analytical - db_numerical):.2e}')
print(f'  Gradients match!' if abs(dw_analytical - dw_numerical) < 1e-4 else '  Check your gradients!')

### Exercise 2: Try Your Own Learning Rate
嘗試一個介於 0.01 和 0.1 之間的學習率（例如 0.05），觀察結果。

In [ ]:
# Exercise 2: Try your own learning rate
# TODO: Change the learning rate and observe the behavior
my_lr = 0.05  # <-- Try changing this value!
my_epochs = 100

w_my, b_my, l_my = gradient_descent(X, y, lr=my_lr, epochs=my_epochs)

print(f'lr = {my_lr}, epochs = {my_epochs}')
print(f'Final: w = {w_my[-1]:.4f}, b = {b_my[-1]:.4f}, MSE = {l_my[-1]:.4f}')

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(l_my, 'b-', linewidth=2)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('MSE')
ax1.set_title(f'Loss Curve (lr={my_lr})')
ax1.grid(True, alpha=0.3)

ax2.contourf(W_grid, B_grid, L_grid, levels=25, cmap=cm.viridis, alpha=0.3)
ax2.contour(W_grid, B_grid, L_grid, levels=25, cmap=cm.viridis, alpha=0.5)
ax2.plot(w_my, b_my, 'r.-', markersize=3, linewidth=1.5)
ax2.plot(w_my[0], b_my[0], 'go', markersize=10)
ax2.plot(w_my[-1], b_my[-1], 'r*', markersize=15)
ax2.set_xlabel('w'); ax2.set_ylabel('b')
ax2.set_title(f'Trajectory (lr={my_lr})')
ax2.set_xlim(w_range); ax2.set_ylim(b_range)
ax2.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

### Exercise 3: Feature Scaling Effect
將特徵 X 標準化 (Standardize) 後再跑梯度下降，觀察等高線形狀和收斂速度的變化。

提示：$X_{\text{scaled}} = \frac{X - \mu_X}{\sigma_X}$

In [ ]:
# Exercise 3: Feature Scaling
# TODO: Standardize X and re-run gradient descent
X_scaled = (X - X.mean()) / X.std()

# Re-run gradient descent with scaled features
w_scaled, b_scaled, l_scaled = gradient_descent(X_scaled, y, lr=0.1, epochs=100)

print(f'Scaled GD: w={w_scaled[-1]:.4f}, b={b_scaled[-1]:.4f}, MSE={l_scaled[-1]:.4f}')
print(f'Note: w and b have different meaning after scaling!')
print(f'  After un-scaling: w_original = w_scaled / std(X) = {w_scaled[-1]/X.std():.4f}')
print(f'  After un-scaling: b_original = b_scaled - w_scaled * mean(X)/std(X) = '
      f'{b_scaled[-1] - w_scaled[-1]*X.mean()/X.std():.4f}')

### Exercise 4: Normal Equation Implementation
實作正規方程 (Normal Equation)，與梯度下降的結果比較。

$$\boldsymbol{\theta}^* = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}$$

In [ ]:
# Exercise 4: Normal Equation
# Build design matrix [X, 1]
X_design = np.column_stack([X, np.ones(n_samples)])

# Normal equation: theta = (X^T X)^(-1) X^T y
theta_normal = np.linalg.inv(X_design.T @ X_design) @ X_design.T @ y
w_normal, b_normal = theta_normal[0], theta_normal[1]

print('=== Normal Equation vs Gradient Descent ===')
print(f'{"":<20} {"Normal Eq":>12} {"GD (lr=0.01)":>12} {"True":>12}')
print(f'{"w":<20} {w_normal:>12.6f} {w_final:>12.6f} {w_true:>12.6f}')
print(f'{"b":<20} {b_normal:>12.6f} {b_final:>12.6f} {b_true:>12.6f}')
print(f'{"MSE":<20} {compute_mse(X, y, w_normal, b_normal):>12.6f} {loss_final:>12.6f}')

---

## Summary
## 本週總結

### Key Concepts Learned:
1. **Linear Regression** fits a line $\hat{y} = wx + b$ to minimize prediction error
2. **MSE Loss** measures the average squared residual; its bowl-shaped surface guarantees a unique optimum
3. **Gradient Descent** iteratively updates parameters along the steepest descent direction
4. **Learning Rate** is critical: too large = divergence, too small = slow, just right = stable convergence
5. **Loss Landscape Visualization** builds intuition about optimization geometry
6. **BGD vs SGD vs Mini-batch** trade off accuracy, speed, and stability
7. **Residual Analysis** validates model assumptions
8. **Polynomial Regression** can overfit if degree is too high

### Next Week Preview:
**Week 5: Classification** - Logistic Regression, Decision Boundaries, ROC/PR Curves
- We'll see how gradient descent is used for classification tasks!
- Different loss function (Cross-Entropy) but same optimization principle